In [ ]:
%pip install -q "kaggle-environments==1.32.7"

In [ ]:
!git clone https://github.com/seabreezeunderthepalmtree/hungry-geese.git
%cd hungry-geese

Cloning into 'hungry-geese'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 48 (delta 25), reused 40 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 44.98 KiB | 2.37 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/hungry-geese/hungry-geese


In [ ]:
!git pull

Already up to date.


In [ ]:
#@title 自选四个模型进行对局

import torch
from kaggle_environments import make

from agent import Agent
from model import HungryGeeseActorCritic


# 自己填写四个模型路径。
model_paths = [
    "/content/hungry-geese/models/model_000080.pt",
    "/content/hungry-geese/models/model_000080.pt",
    "/content/hungry-geese/models/model_000080.pt",
    "/content/hungry-geese/models/model_000080.pt",
]

agents = []

for model_path in model_paths:
    checkpoint = torch.load(
        model_path,
        map_location="cpu",
        weights_only=True,
    )

    # 兼容原始权重和包含 optimizer state 的训练 checkpoint。
    if (
        isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
    ):
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    model = HungryGeeseActorCritic()
    model.load_state_dict(state_dict)
    model.eval()

    agents.append(Agent(model))

    print(f"Loaded: {model_path}")


env = make(
    "hungry_geese",
    configuration={
        "episodeSteps": 200,
    },
    debug=True,
)

env.run(agents)

env.render(
    mode="ipython",
    width=800,
    height=700,
)

Loaded: /content/hungry-geese/models/model_000080.pt
Loaded: /content/hungry-geese/models/model_000080.pt
Loaded: /content/hungry-geese/models/model_000080.pt
Loaded: /content/hungry-geese/models/model_000080.pt
Goose Collision: WEST


In [ ]:
#@title 持续生成对局并训练，使用 Early Stopping

from pathlib import Path
import json
import re
import subprocess
import sys


# ========================
# 可修改参数
# ========================

PROJECT_DIR = Path("/content/hungry-geese")

# 每个 iteration 生成的训练对局数。
TRAIN_GAMES = 32

# 每隔多少个 iteration 评估一次。
EVALUATE_EVERY = 10

# 每次评估使用的对局数。
EVALUATION_GAMES = 32

# 连续多少次评估没有改善后停止。
# PATIENCE=5 且 EVALUATE_EVERY=10，
# 表示至少观察约 50 个训练 iteration。
PATIENCE = 5

MIN_DELTA = 0.03

TRAINING_GAMEPLAYS_DIR = PROJECT_DIR / "gameplays"
EVALUATION_GAMEPLAYS_DIR = PROJECT_DIR / "evaluation_gameplays"
TRAINING_LOGS_DIR = PROJECT_DIR / "training_logs"
MODELS_DIR = PROJECT_DIR / "models"

PYTHON = sys.executable

RANK_REWARDS = {
    1: 1.0,
    2: 1.0 / 3.0,
    3: -1.0 / 3.0,
    4: -1.0,
}


def latest_model_id():
    """返回最新模型编号。"""
    pattern = re.compile(r"model_(\d+)\.(?:pt|pth)$")

    model_ids = [
        int(match.group(1))
        for path in MODELS_DIR.glob("model_*")
        if (match := pattern.fullmatch(path.name))
    ]

    if not model_ids:
        return None

    return max(model_ids)


def evaluate_latest_model():
    """让最新模型对战三个 SimpleAgent，并返回平均排名奖励。"""
    model_id = latest_model_id()

    if model_id is None:
        raise RuntimeError("没有找到模型")

    output_directory = (
        EVALUATION_GAMEPLAYS_DIR
        / f"model_{model_id:06d}"
    )

    existing_files = set(
        output_directory.glob("game_*.json")
    )

    subprocess.run(
        [
            PYTHON,
            "generate.py",
            "--models-dir",
            str(MODELS_DIR),
            "--gameplays-dir",
            str(EVALUATION_GAMEPLAYS_DIR),
            "--games",
            str(EVALUATION_GAMES),
            "--simple-agent-probability",
            "1.0",
        ],
        cwd=PROJECT_DIR,
        check=True,
    )

    generated_files = sorted(
        set(output_directory.glob("game_*.json"))
        - existing_files
    )

    if not generated_files:
        raise RuntimeError("没有生成 evaluation replay")

    ranking_rewards = []

    for replay_path in generated_files:
        with replay_path.open(
            "r",
            encoding="utf-8",
        ) as replay_file:
            replay = json.load(replay_file)

        trainable_players = replay["ppo"]["trainable_players"]

        # simple-agent-probability=1.0 时，
        # generate.py 会确保恰好保留一个模型玩家。
        if len(trainable_players) != 1:
            raise RuntimeError(
                f"{replay_path} 中的模型玩家数量不是 1"
            )

        model_player = trainable_players[0]
        final_scores = replay["rewards"]
        model_score = final_scores[model_player]

        # 并列取较低名次。
        rank = sum(
            score >= model_score
            for score in final_scores
        )

        ranking_rewards.append(
            RANK_REWARDS[rank]
        )

    mean_score = (
        sum(ranking_rewards)
        / len(ranking_rewards)
    )

    # 避免显示 -0.0000。
    if abs(mean_score) < 1e-12:
        mean_score = 0.0

    return mean_score


starting_model_id = latest_model_id()

print(
    "Starting from:",
    (
        f"model_{starting_model_id:06d}"
        if starting_model_id is not None
        else "random initialization"
    ),
)

best_score = float("-inf")
best_model_id = None
evaluations_without_improvement = 0
iteration = 0


while evaluations_without_improvement < PATIENCE:
    iteration += 1

    print()
    print("=" * 60)
    print(f"Iteration {iteration}")
    print("=" * 60)

    # 1. 使用当前最新模型生成训练对局。
    subprocess.run(
        [
            PYTHON,
            "generate.py",
            "--models-dir",
            str(MODELS_DIR),
            "--gameplays-dir",
            str(TRAINING_GAMEPLAYS_DIR),
            "--games",
            str(TRAIN_GAMES),
        ],
        cwd=PROJECT_DIR,
        check=True,
    )

    # 2. 训练并保存下一个编号的模型。
    # train.py 会自动保存：
    # models/model_XXXXXX.pt
    # training_logs/model_XXXXXX.json
    subprocess.run(
        [
            PYTHON,
            "train.py",
            "--models-dir",
            str(MODELS_DIR),
            "--gameplays-dir",
            str(TRAINING_GAMEPLAYS_DIR),
            "--logs-dir",
            str(TRAINING_LOGS_DIR),
            "--device",
            "auto",
        ],
        cwd=PROJECT_DIR,
        check=True,
    )

    current_model_id = latest_model_id()

    print(
        f"Finished model_{current_model_id:06d}"
    )
    print(
        "Training log:",
        TRAINING_LOGS_DIR
        / f"model_{current_model_id:06d}.json",
    )

    # 3. 每 10 个 iteration 才评估一次。
    if iteration % EVALUATE_EVERY != 0:
        remaining = EVALUATE_EVERY - (
            iteration % EVALUATE_EVERY
        )

        print(
            f"Evaluation skipped; "
            f"next evaluation in {remaining} iteration(s)."
        )

        continue

    print()
    print(
        f"Evaluating model_{current_model_id:06d} "
        f"with {EVALUATION_GAMES} games..."
    )

    current_score = evaluate_latest_model()

    print(
        f"Model {current_model_id:06d}: "
        f"evaluation score = {current_score:.4f}"
    )

    # 4. Early stopping。
    if current_score > best_score + MIN_DELTA:
        best_score = current_score
        best_model_id = current_model_id
        evaluations_without_improvement = 0

        print(
            f"New best model: model_{best_model_id:06d}, "
            f"score={best_score:.4f}"
        )
    else:
        evaluations_without_improvement += 1

        print(
            "No sufficient improvement: "
            f"{evaluations_without_improvement}/{PATIENCE} "
            "evaluation rounds"
        )


print()
print("Early stopping triggered.")

if best_model_id is not None:
    print(f"Best model: model_{best_model_id:06d}.pt")
    print(f"Best evaluation score: {best_score:.4f}")

Starting from: model_000040

Iteration 1
Finished model_000041
Training log: /content/hungry-geese/training_logs/model_000041.json
Evaluation skipped; next evaluation in 9 iteration(s).

Iteration 2
Finished model_000042
Training log: /content/hungry-geese/training_logs/model_000042.json
Evaluation skipped; next evaluation in 8 iteration(s).

Iteration 3
Finished model_000043
Training log: /content/hungry-geese/training_logs/model_000043.json
Evaluation skipped; next evaluation in 7 iteration(s).

Iteration 4
Finished model_000044
Training log: /content/hungry-geese/training_logs/model_000044.json
Evaluation skipped; next evaluation in 6 iteration(s).

Iteration 5
Finished model_000045
Training log: /content/hungry-geese/training_logs/model_000045.json
Evaluation skipped; next evaluation in 5 iteration(s).

Iteration 6
Finished model_000046
Training log: /content/hungry-geese/training_logs/model_000046.json
Evaluation skipped; next evaluation in 4 iteration(s).

Iteration 7
Finished mo